# Write Derived Results Back To Raw Binary Files

This notebook focuses on the write side of xarray-binfile. The workflow is:

- open a binary dataset lazily with xarray and Dask
- build a derived result without immediately loading everything
- compute a small summary when you need an actual value
- write a derived array back to `.bin` files with `.binary_engine.to_file(...)`
- read the result again to verify the round trip

```{note}
Raw binary output is useful when you must match an existing external convention. If you control the output format, prefer NetCDF or Zarr for portability, metadata preservation, and interoperability across tools.
```

In [ ]:
import pathlib
import re
import tempfile
from dataclasses import dataclass

import numpy as np
import xarray as xr

from xarray_binfile.tutorial import DatasetGenerator, FileSpecsGetter
from xarray_binfile.write import WriteSpecs

input_directory_holder = tempfile.TemporaryDirectory()
output_directory_holder = tempfile.TemporaryDirectory()
input_directory = pathlib.Path(input_directory_holder.name)
output_directory = pathlib.Path(output_directory_holder.name)
input_directory, output_directory

In [ ]:
base_coords = {
    "x": np.linspace(0.0, 2.0, num=24, dtype=np.float32),
    "y": np.linspace(-1.0, 1.0, num=18, dtype=np.float32),
    "z": np.linspace(0.0, 1.5, num=12, dtype=np.float32),
}

input_specs_getter = FileSpecsGetter(
    base_coords=base_coords,
    dtype=np.float32,
    filename_template="{name}-{digits:04}.bin",
    filename_regex=re.compile(r"(?P<name>\w+)-(?P<digits>\d{4})\.bin"),
)

dataset_generator = DatasetGenerator(input_specs_getter.reader)
filenames = (
    input_specs_getter.filename_template.format(name=name, digits=step)
    for name in ("ux", "uy", "uz")
    for step in range(8)
)
source_dataset = dataset_generator(map(pathlib.Path, filenames))
source_dataset.binary_engine.to_file(input_specs_getter.writer, input_directory)
sorted(path.name for path in input_directory.glob("*.bin"))[:6]

In [ ]:
lazy_dataset = xr.open_mfdataset(
    sorted(input_directory.glob("*.bin")),
    engine="binfile",
    read_specs_getter=input_specs_getter.reader,
    chunks={"x": 6, "y": 6, "z": 4, "time": 2},
    parallel=True,
)
lazy_dataset

In [ ]:
speed = np.sqrt(
    lazy_dataset["ux"] ** 2 + lazy_dataset["uy"] ** 2 + lazy_dataset["uz"] ** 2
).rename("speed")
speed

The derived `speed` array is still lazy. You can compute a lightweight summary first, then decide whether to persist a full result.

In [ ]:
speed_time_series = speed.mean(dim=("x", "y", "z")).compute()
speed_time_series

## Custom write specifications

`FileSpecsGetter.writer` is useful when your output convention matches the tutorial helper. If you need a different layout, implement a callable compatible with `WriteSpecsGetterProtocol`.

In [ ]:
@dataclass(frozen=True)
class DerivedWriteSpecsGetter:
    prefix: str = "derived"
    base_dims: tuple[str, ...] = ("x", "y", "z")

    def __call__(self, data_array: xr.DataArray):
        for time_value in data_array.coords["time"].values:
            timestep = int(time_value)
            yield WriteSpecs(
                filename=f"{self.prefix}-{data_array.name}-{timestep:04}.bin",
                sub_array=data_array.sel(time=timestep).transpose(
                    *self.base_dims, missing_dims="raise"
                ),
            )


derived_write_specs_getter = DerivedWriteSpecsGetter()
derived_write_specs_getter

In [ ]:
speed.binary_engine.to_file(derived_write_specs_getter, output_directory)
sorted(path.name for path in output_directory.glob("*.bin"))[:6]

Writing triggers the actual per-file materialization of each selected slice. That is often exactly what you want when handing data back to an external solver or post-processing pipeline.

In [ ]:
output_specs_getter = FileSpecsGetter(
    base_coords=base_coords,
    dtype=np.float32,
    filename_template="derived-{name}-{digits:04}.bin",
    filename_regex=re.compile(r"derived-(?P<name>\w+)-(?P<digits>\d{4})\.bin"),
)

roundtrip_dataset = xr.open_mfdataset(
    sorted(output_directory.glob("derived-*.bin")),
    engine="binfile",
    read_specs_getter=output_specs_getter.reader,
    chunks={"time": 2},
    parallel=True,
)
roundtrip_dataset

In [ ]:
computed_speed = speed.compute().to_dataset()
xr.testing.assert_allclose(roundtrip_dataset.compute(), computed_speed)
computed_speed["speed"].isel(time=0, z=0).plot(cmap="magma", robust=True)

For general exchange and long-term storage, use xarray's standard formats when possible: `Dataset.to_netcdf(...)` and `Dataset.to_zarr(...)` preserve metadata and make collaboration easier. Use raw binary output when you need strict compatibility with an external file convention.